# Small Dataset Tuning

小規模 HDF5 を使って、Jupyter 上で設定を変えながら学習と診断を行う notebook です。
Notebook で作る図、checkpoint、JSON は本番 run と混ぜず、`NOTEBOOK_ROOT` 以下に保存します。

主に触る場所は `DATASET_KEY`、`NOTEBOOK_ROOT`、学習設定です。`flat200` と `flat500` は動作確認、`flat2000` は軽い比較、`flat20000` と `flat50000` は統計を見たい実験に使います。


## 1. 初期化

リポジトリを `sys.path` に入れ、学習本体と解析関数を import します。Jupyter の current directory がリポジトリでない場合だけ、既定の repo path にフォールバックします。

In [ ]:
%matplotlib inline

from pathlib import Path
import json
import random
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import clear_output, display

repo = Path.cwd()
if not (repo / "src" / "talesd_gnn_reconstruction").exists():
    repo = Path("/ceph/sharedfs/work/SATORI/ikomae/src/talesd_gnn_reconstruction")
sys.path.insert(0, str(repo / "src"))

from talesd_gnn_reconstruction.dataset import H5GraphDataset, fit_scalers
from talesd_gnn_reconstruction.diagnostics import save_learning_progress
from talesd_gnn_reconstruction.feature_analysis import (
    save_feature_group_importance,
    save_input_distributions,
)
from talesd_gnn_reconstruction.model import PhysicsTaleSdGNN, TaleSdGNN
from talesd_gnn_reconstruction.train import (
    _batch_to_device,
    _make_graph_loader,
    _quality_prediction_loss,
    _reconstruction_training_loss,
    _split_model_output,
    _target_scaler_tensors,
    resolve_device,
    split_indices,
    split_indices_by_stratified_source_path,
)

print(f"repo: {repo}")
NOTEBOOK_ROOT = Path("/dicos_ui_home/ikomae/work/gnn/outputs/talesd_gnn_reconstruction/notebook")
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)
print(f"notebook output root: {NOTEBOOK_ROOT}")


## 2. データセットを選ぶ

`DATASET_KEY` を変えるだけで、使う HDF5 を切り替えます。ここでは HDF5 shard を持つディレクトリを指定します。存在しない場合は、この kernel からその path が見えていないという意味です。

In [ ]:
GRAPH_SETS = {
    "flat200": "/dicos_ui_home/ikomae/work/gnn/graphs/flat200",
    "flat500": "/dicos_ui_home/ikomae/work/gnn/graphs/flat500",
    "flat2000": "/dicos_ui_home/ikomae/work/gnn/graphs/flat2000",
    "flat20000": "/dicos_ui_home/ikomae/work/gnn/graphs/flat20000",
    "flat50000": "/dicos_ui_home/ikomae/work/gnn/graphs/flat50000",
}

DATASET_KEY = "flat2000"
graph_dir = Path(GRAPH_SETS[DATASET_KEY]).expanduser()
graph_files = sorted(graph_dir.glob("*.h5")) if graph_dir.exists() else []

print(f"dataset: {DATASET_KEY}")
print(f"graph_dir: {graph_dir}")
print(f"h5 shards: {len(graph_files)}")
for path in graph_files[:10]:
    print(f"  {path.name}")
if not graph_files:
    print("この kernel から graph shard が見えていません。DiCOS App の実行環境と path を確認してください。")

## 3. 入力パラメーター分布を描く

学習前に、ノード特徴量、エッジ特徴量、パルス特徴量、波形値、target の分布を確認します。全件を読むと重い場合があるため、まずは `max_graphs` を制限します。出力は `NOTEBOOK_ROOT / "diagnostics"` 以下の PDF と JSON です。

In [ ]:
distribution_dir = NOTEBOOK_ROOT / "diagnostics" / DATASET_KEY / "input_distributions"

RUN_INPUT_DISTRIBUTIONS = False
if RUN_INPUT_DISTRIBUTIONS:
    distribution_result = save_input_distributions(
        graph_dir,
        distribution_dir,
        max_graphs=100000,
        max_values_per_feature=200000,
        seed=12345,
        show_progress=True,
    )
    print(json.dumps(distribution_result, indent=2)[:8000])
else:
    print("RUN_INPUT_DISTRIBUTIONS=True にすると分布図を作成します。")
    print(f"output: {distribution_dir}")

## 4. 学習設定

Notebook 用の学習設定です。標準設定は `physics + cnn-gru + quality` です。ここで変更した値を下の学習セルが直接使います。

In [ ]:
run_name = f"notebook_{DATASET_KEY}_reco_quality"
run_dir = NOTEBOOK_ROOT / "runs" / run_name
checkpoint = run_dir / "checkpoints" / f"{run_name}.pt"

train_kwargs = dict(
    graphs_path=str(graph_dir),
    output_path=str(checkpoint),
    epochs=16,
    batch_size=256,
    learning_rate=3e-4,
    weight_decay=1e-4,
    hidden_dim=192,
    num_layers=5,
    dropout=0.05,
    lr_scheduler="reduce-on-plateau",
    lr_factor=0.5,
    lr_patience=2,
    early_stopping_patience=12,
    early_stopping_min_epochs=32,
    model_architecture="physics",
    readout_heads=4,
    classification_arch="enhanced",
    detector_embedding_dim=0,
    waveform_encoder="cnn-gru",
    waveform_embedding_dim=64,
    waveform_transformer_heads=4,
    waveform_transformer_layers=1,
    loss_mode="physics",
    energy_loss_weight=1.2,
    core_loss_weight=1.0,
    direction_loss_weight=1.4,
    core_loss_scale_km=0.12,
    angular_loss_scale_deg=1.0,
    val_fraction=0.05,
    test_fraction=0.10,
    split_mode="source-stratified",
    seed=12345,
    device="auto",
    sample_cache_size=0,
    max_graphs=None,
    particle_filter="all",
    pin_memory=None,
    num_workers=4,
    preprocess_workers=8,
    prefetch_factor=2,
    persistent_workers=False,
    collate_backend="cpp",
    collate_threads=1,
    training_task="reconstruction",
    mass_classification=False,
    quality_prediction=True,
    quality_loss_weight=0.2,
    quality_angular_scale_deg=1.0,
    quality_core_scale_km=0.05,
    quality_energy_scale=0.10,
    error_prediction=False,
    error_angular_scale_deg=1.0,
    error_core_scale_km=0.05,
    error_energy_scale=0.10,
    nll_loss_weight=0.0,
    nll_sigma_energy_floor=0.01,
    nll_sigma_angle_floor_deg=0.05,
    nll_sigma_core_floor_km=0.005,
    show_progress=True,
    save_diagnostics=True,
    update_learning_curve_each_epoch=True,
    best_diagnostics=True,
    best_diagnostic_max_graphs=20000,
    diagnostic_energy_bin_width=0.1,
    diagnostic_min_bin_count=1000,
)

print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in train_kwargs.items()}, indent=2)[:12000])

## 5. 学習を実行する

`RUN_TRAINING=True` にすると、上の設定でこのセル内の学習ループを実行します。図と checkpoint は `NOTEBOOK_ROOT / "runs"` 以下に保存します。

`LIVE_LOSS_PLOT=True` の場合は、各 epoch が終わるたびに notebook のinline の出力セル内で loss curve を更新します。batch ごとの描画は重いため行いません。

In [ ]:
RUN_TRAINING = False
LIVE_LOSS_PLOT = True


def plot_history(history):
    if not history:
        print("history is empty")
        return
    epochs = [row["epoch"] for row in history]
    train_loss = [row.get("train_loss") for row in history]
    val_loss = [row.get("val_loss") for row in history]
    finite_val = [(i, value) for i, value in enumerate(val_loss) if value is not None]
    best_index = min(finite_val, key=lambda item: item[1])[0] if finite_val else None

    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.plot(epochs, train_loss, marker="o", linewidth=1.4, label="train loss")
    ax.plot(epochs, val_loss, marker="o", linewidth=1.4, label="validation loss")
    if best_index is not None:
        ax.axvline(epochs[best_index], color="0.35", linestyle="--", linewidth=1.0, label=f"best epoch {epochs[best_index]}")
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.set_title("training progress")
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend(frameon=False)
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def mean_loss(values):
    return float(np.mean(values)) if values else float("nan")


if RUN_TRAINING:
    random.seed(train_kwargs["seed"])
    np.random.seed(train_kwargs["seed"])
    torch.manual_seed(train_kwargs["seed"])

    device = resolve_device(train_kwargs["device"])
    pin_memory = device.startswith("cuda") if train_kwargs["pin_memory"] is None else bool(train_kwargs["pin_memory"])
    output = Path(train_kwargs["output_path"])
    output.parent.mkdir(parents=True, exist_ok=True)

    dataset = H5GraphDataset(
        train_kwargs["graphs_path"],
        require_target=True,
        require_particle_label=False,
        cache_size=train_kwargs["sample_cache_size"],
        load_node_positions=False,
        load_attrs=False,
        load_particle_label=False,
        load_detector_lids=False,
        max_graphs=train_kwargs["max_graphs"],
        particle_filter=train_kwargs["particle_filter"],
        show_progress=train_kwargs["show_progress"],
    )

    if train_kwargs["split_mode"] == "source-stratified":
        split = split_indices_by_stratified_source_path(
            dataset,
            val_fraction=train_kwargs["val_fraction"],
            test_fraction=train_kwargs["test_fraction"],
            seed=train_kwargs["seed"],
            show_progress=train_kwargs["show_progress"],
            workers=train_kwargs["preprocess_workers"],
        )
    elif train_kwargs["split_mode"] == "event":
        split = split_indices(
            len(dataset),
            val_fraction=train_kwargs["val_fraction"],
            test_fraction=train_kwargs["test_fraction"],
            seed=train_kwargs["seed"],
        )
    else:
        raise ValueError("この notebook では split_mode='source-stratified' または 'event' を使います。")

    train_indices = split["train"]
    val_indices = split["val"]
    test_indices = split["test"]
    print(f"split: train={len(train_indices)} val={len(val_indices)} test={len(test_indices)}")

    scalers = fit_scalers(
        dataset,
        sorted(train_indices),
        show_progress=train_kwargs["show_progress"],
        workers=train_kwargs["preprocess_workers"],
    )

    first = dataset[train_indices[0]]
    graph_target_dim = int(first["target"].shape[0])
    target_dim = graph_target_dim
    model_kwargs = dict(
        node_dim=first["node_features"].shape[1],
        edge_dim=first["edge_features"].shape[1],
        pulse_dim=max(first["pulse_features"].shape[1] - 1, 0),
        waveform_channels=(
            first["waveform_features"].shape[1]
            if first.get("waveform_features") is not None and first["waveform_features"].ndim == 3
            else 0
        ),
        waveform_length=(
            first["waveform_features"].shape[2]
            if first.get("waveform_features") is not None and first["waveform_features"].ndim == 3
            else 0
        ),
        waveform_encoder=train_kwargs["waveform_encoder"],
        waveform_embedding_dim=train_kwargs["waveform_embedding_dim"],
        waveform_transformer_heads=train_kwargs["waveform_transformer_heads"],
        waveform_transformer_layers=train_kwargs["waveform_transformer_layers"],
        target_dim=target_dim,
        classification_dim=1 if train_kwargs["mass_classification"] else 0,
        quality_dim=1 if train_kwargs["quality_prediction"] else 0,
        error_dim=3 if train_kwargs["error_prediction"] else 0,
        hidden_dim=train_kwargs["hidden_dim"],
        num_layers=train_kwargs["num_layers"],
        dropout=train_kwargs["dropout"],
        classification_arch=train_kwargs["classification_arch"],
        detector_lids=[],
        detector_embedding_dim=0,
    )

    if train_kwargs["model_architecture"] == "physics":
        model = PhysicsTaleSdGNN(**model_kwargs, readout_heads=train_kwargs["readout_heads"]).to(device)
    elif train_kwargs["model_architecture"] == "baseline":
        model = TaleSdGNN(**model_kwargs).to(device)
    else:
        raise ValueError("model_architecture must be 'physics' or 'baseline'")

    train_loader = _make_graph_loader(
        dataset,
        train_indices,
        scalers=scalers,
        batch_size=train_kwargs["batch_size"],
        shuffle=True,
        require_target=True,
        num_workers=train_kwargs["num_workers"],
        prefetch_factor=train_kwargs["prefetch_factor"],
        seed=train_kwargs["seed"],
        pin_memory=pin_memory,
        persistent_workers=train_kwargs["persistent_workers"],
        collate_backend=train_kwargs["collate_backend"],
        collate_threads=train_kwargs["collate_threads"],
    )
    val_loader = _make_graph_loader(
        dataset,
        val_indices,
        scalers=scalers,
        batch_size=train_kwargs["batch_size"],
        shuffle=False,
        require_target=True,
        num_workers=train_kwargs["num_workers"],
        prefetch_factor=train_kwargs["prefetch_factor"],
        seed=train_kwargs["seed"],
        pin_memory=pin_memory,
        persistent_workers=train_kwargs["persistent_workers"],
        collate_backend=train_kwargs["collate_backend"],
        collate_threads=train_kwargs["collate_threads"],
    )

    target_mean, target_std = _target_scaler_tensors(scalers, device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=train_kwargs["learning_rate"], weight_decay=train_kwargs["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=train_kwargs["lr_factor"],
        patience=train_kwargs["lr_patience"],
    )

    history = []
    best_val = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    started = time.perf_counter()

    for epoch in range(1, train_kwargs["epochs"] + 1):
        model.train()
        train_losses = []
        train_reco_losses = []
        train_quality_losses = []

        for batch_cpu in train_loader:
            batch = _batch_to_device(batch_cpu, device, non_blocking=pin_memory)
            pred_all = model(batch)
            pred, mass_logit, quality_logit, error_raw = _split_model_output(
                pred_all,
                target_dim,
                train_kwargs["mass_classification"],
                train_kwargs["quality_prediction"],
                train_kwargs["error_prediction"],
            )
            reco_loss, _components = _reconstruction_training_loss(
                pred,
                batch["y"],
                error_raw,
                mode=train_kwargs["loss_mode"],
                target_mean=target_mean,
                target_std=target_std,
                energy_weight=train_kwargs["energy_loss_weight"],
                core_weight=train_kwargs["core_loss_weight"],
                direction_weight=train_kwargs["direction_loss_weight"],
                core_scale_km=train_kwargs["core_loss_scale_km"],
                angular_loss_scale_deg=train_kwargs["angular_loss_scale_deg"],
                nll_loss_weight=train_kwargs["nll_loss_weight"],
                error_angular_scale_deg=train_kwargs["error_angular_scale_deg"],
                error_core_scale_km=train_kwargs["error_core_scale_km"],
                error_energy_scale=train_kwargs["error_energy_scale"],
                nll_sigma_energy_floor=train_kwargs["nll_sigma_energy_floor"],
                nll_sigma_angle_floor_deg=train_kwargs["nll_sigma_angle_floor_deg"],
                nll_sigma_core_floor_km=train_kwargs["nll_sigma_core_floor_km"],
            )
            loss = reco_loss
            quality_loss = None
            if train_kwargs["quality_prediction"]:
                quality_loss = _quality_prediction_loss(
                    quality_logit,
                    pred,
                    batch["y"],
                    target_mean=target_mean,
                    target_std=target_std,
                    angular_scale_deg=train_kwargs["quality_angular_scale_deg"],
                    core_scale_km=train_kwargs["quality_core_scale_km"],
                    energy_scale=train_kwargs["quality_energy_scale"],
                )
                loss = loss + train_kwargs["quality_loss_weight"] * quality_loss

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            train_losses.append(float(loss.detach().cpu()))
            train_reco_losses.append(float(reco_loss.detach().cpu()))
            if quality_loss is not None:
                train_quality_losses.append(float(quality_loss.detach().cpu()))

        model.eval()
        val_losses = []
        val_reco_losses = []
        val_quality_losses = []
        with torch.no_grad():
            for batch_cpu in val_loader:
                batch = _batch_to_device(batch_cpu, device, non_blocking=pin_memory)
                pred_all = model(batch)
                pred, mass_logit, quality_logit, error_raw = _split_model_output(
                    pred_all,
                    target_dim,
                    train_kwargs["mass_classification"],
                    train_kwargs["quality_prediction"],
                    train_kwargs["error_prediction"],
                )
                reco_loss, _components = _reconstruction_training_loss(
                    pred,
                    batch["y"],
                    error_raw,
                    mode=train_kwargs["loss_mode"],
                    target_mean=target_mean,
                    target_std=target_std,
                    energy_weight=train_kwargs["energy_loss_weight"],
                    core_weight=train_kwargs["core_loss_weight"],
                    direction_weight=train_kwargs["direction_loss_weight"],
                    core_scale_km=train_kwargs["core_loss_scale_km"],
                    angular_loss_scale_deg=train_kwargs["angular_loss_scale_deg"],
                    nll_loss_weight=train_kwargs["nll_loss_weight"],
                )
                loss = reco_loss
                quality_loss = None
                if train_kwargs["quality_prediction"]:
                    quality_loss = _quality_prediction_loss(
                        quality_logit,
                        pred,
                        batch["y"],
                        target_mean=target_mean,
                        target_std=target_std,
                        angular_scale_deg=1.0,
                        core_scale_km=0.05,
                        energy_scale=0.10,
                    )
                    loss = loss + train_kwargs["quality_loss_weight"] * quality_loss
                val_losses.append(float(loss.detach().cpu()))
                val_reco_losses.append(float(reco_loss.detach().cpu()))
                if quality_loss is not None:
                    val_quality_losses.append(float(quality_loss.detach().cpu()))

        epoch_row = dict(
            epoch=epoch,
            train_loss=mean_loss(train_losses),
            val_loss=mean_loss(val_losses),
            train_reco_loss=mean_loss(train_reco_losses),
            val_reco_loss=mean_loss(val_reco_losses),
            train_quality_loss=mean_loss(train_quality_losses),
            val_quality_loss=mean_loss(val_quality_losses),
            lr=float(optimizer.param_groups[0]["lr"]),
            elapsed_s=round(time.perf_counter() - started, 3),
        )
        history.append(epoch_row)
        scheduler.step(epoch_row["val_loss"])

        checkpoint_payload = dict(
            model_state=model.state_dict(),
            model_config=model.config,
            scalers={name: scaler.to_dict() for name, scaler in scalers.items()},
            history=history,
            train_indices=np.asarray(train_indices, dtype=np.int64),
            val_indices=np.asarray(val_indices, dtype=np.int64),
            test_indices=np.asarray(test_indices, dtype=np.int64),
            split=dict(n_train=len(train_indices), n_val=len(val_indices), n_test=len(test_indices), split_mode=train_kwargs["split_mode"]),
            runtime=dict(best_epoch=best_epoch, best_val_loss=best_val, notebook=True, **train_kwargs),
        )
        torch.save(checkpoint_payload, output.with_suffix(".last.pt"))

        improved = epoch_row["val_loss"] < best_val
        if improved:
            best_val = epoch_row["val_loss"]
            best_epoch = epoch
            epochs_without_improvement = 0
            checkpoint_payload["runtime"]["best_epoch"] = best_epoch
            checkpoint_payload["runtime"]["best_val_loss"] = best_val
            torch.save(checkpoint_payload, output)
        else:
            epochs_without_improvement += 1

        save_learning_progress(output, history)

        if LIVE_LOSS_PLOT:
            clear_output(wait=True)
            plot_history(history)
        print(json.dumps(epoch_row, indent=2))
        print(f"best_epoch={best_epoch} best_val_loss={best_val:.6g}")
        print(f"checkpoint={output}")

        if (
            train_kwargs["early_stopping_patience"] > 0
            and epoch >= train_kwargs["early_stopping_min_epochs"]
            and epochs_without_improvement >= train_kwargs["early_stopping_patience"]
        ):
            print(f"early stopping at epoch {epoch}")
            break
else:
    print("RUN_TRAINING=True にすると、このセル内の学習ループを実行します。")
    print(f"checkpoint: {checkpoint}")
    print(f"learning curve: {checkpoint}.diagnostics/learning_curve.pdf")


## 6. 特徴量 group の寄与を評価する

学習済み checkpoint に対して、特徴量 group を学習データ平均で置き換え、validation 指標がどれだけ悪化するかを測ります。これは再学習を行わない post-hoc ablation です。`flat50000` でも再学習を何十回も回す必要はありません。

In [ ]:
importance_dir = run_dir / "feature_importance"

RUN_FEATURE_IMPORTANCE = False
if RUN_FEATURE_IMPORTANCE:
    importance_result = save_feature_group_importance(
        graph_dir,
        checkpoint,
        importance_dir,
        split="validation",
        max_graphs=50000,
        batch_size=train_kwargs["batch_size"],
        device=train_kwargs["device"],
        seed=train_kwargs["seed"],
        show_progress=True,
    )
    print(json.dumps(importance_result, indent=2)[:12000])
else:
    print("RUN_FEATURE_IMPORTANCE=True にすると feature group ablation を実行します。")
    print(f"output: {importance_dir}")